# 🗄️ Generador de Scripts SQL para Base de Datos Athena

Este notebook automatiza la generación de scripts SQL de inserción a partir del archivo Excel `tablas_completas.xlsx`.

## 🎯 Objetivo

Transformar los datos estructurados en Excel en scripts SQL optimizados para PostgreSQL, manteniendo la integridad referencial y usando el formato de múltiples valores en un solo INSERT.

## 📋 Hojas procesadas

1. **area** → `public.areas`
2. **escuelas** → `public.escuelas`
3. **carreras** → `public.carreras`
4. **carrera_escuelas** → `public.escuela_carrera`

## ⚡ Características

- ✅ Formato SQL optimizado (un INSERT por tabla con múltiples VALUES)
- ✅ Escape automático de comillas simples
- ✅ Transacciones con BEGIN/COMMIT
- ✅ Encabezados descriptivos por sección
- ✅ Archivo único consolidado

In [1]:
import pandas as pd
from pathlib import Path

## 📦 Dependencias

Importamos las librerías necesarias para:
- `pandas`: Leer archivos Excel y manipular datos
- `pathlib.Path`: Manejar rutas de archivos de manera multiplataforma

## 🔧 Clase Extractor

Esta clase se encarga de extraer datos del Excel y generar scripts SQL optimizados.

### Métodos principales:

1. **`__init__(ruta_excel, ruta_sql_output)`**
   - Inicializa el extractor cargando todas las hojas del Excel
   - Almacena las rutas de entrada y salida

2. **`extract()`**
   - Procesa todas las hojas del Excel usando un switch case
   - Llama al método extractor correspondiente para cada hoja
   - Recopila todos los DML y guarda el archivo final

3. **Métodos privados de extracción:**
   - `_extract_areas()`: Genera INSERT para áreas académicas
   - `_extract_escuelas()`: Genera INSERT para escuelas del IPN
   - `_extract_carreras()`: Genera INSERT para carreras con objetivos
   - `_extract_escuela_carrera()`: Genera INSERT para relación M:N escuela-carrera

4. **`_save_sql(sql_content_list)`**
   - Une todos los INSERTs en un solo archivo
   - Agrega encabezado con metadata
   - Guarda el archivo final

### Formato SQL generado:

Cada tabla usa el formato optimizado:
```sql
INSERT INTO tabla (columna1, columna2) VALUES
    (valor1, valor2),
    (valor3, valor4);
```
Esto es más eficiente que múltiples INSERTs individuales.

In [ ]:
class Extractor:
    def __init__(self, ruta_excel: str, ruta_sql_output: str):
        self.ruta_excel = Path(ruta_excel)
        self.ruta_sql_output = Path(ruta_sql_output)
        self.data = pd.read_excel(self.ruta_excel, sheet_name=None)
        
        print(f"📊 Excel cargado: {self.ruta_excel}")
        print(f"📁 Output SQL: {self.ruta_sql_output}")
        print(f"📋 Hojas disponibles: {list(self.data.keys())}")
    
    def extract(self) -> str:
        all_dml = []
        for sheet_name in self.data.keys():
            match sheet_name.lower():
                case "area":
                    dml_area = self._extract_areas()
                    all_dml.append(dml_area)
                
                case "escuelas":
                    dml_escuelas = self._extract_escuelas()
                    all_dml.append(dml_escuelas)
                
                case "carreras":
                    dml_carreras = self._extract_carreras()
                    all_dml.append(dml_carreras)
                
                case "carrera_escuelas":
                    dml_escuela_carrera = self._extract_escuela_carrera()
                    all_dml.append(dml_escuela_carrera)
                
                case _:
                    raise ValueError(f"Hoja '{sheet_name}' no tiene extractor implementado")

        self._save_sql(all_dml)
    
    def _extract_areas(self) -> str:
        df = self.data['area']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE ÁREAS ACADÉMICAS",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.areas (area_id, nom_area) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            area_id = row['id_area']
            nombre_area = row['nombre_area'].replace("'", "''")
            values.append(f"    ({area_id}, '{nombre_area}')")
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _extract_escuelas(self) -> str:
        df = self.data['escuelas']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE ESCUELAS",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.escuelas (escuela_id, nom_escuela) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            escuela_id = row['id_escuela']
            nombre_escuela = row['nombre_escuela'].replace("'", "''")
            values.append(f"    ({escuela_id}, '{nombre_escuela}')")
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _extract_carreras(self) -> str:
        df = self.data['carreras']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE CARRERAS",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.carreras (carrera_id, area_id, nom_carrera, origina_objetivo, objetivo, pregunta_objetivo) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            carrera_id = row['id_carrera']
            area_id = row['id_area']
            nom_carrera = row['nombre_carrera'].replace("'", "''")
            objetivo_resumido = row['objetivo_resumido'].replace("'", "''")
            pregunta_objetivo = row['pregunta'].replace("'", "''")
            origina_objetivo = row['objetivo'].replace("'", "''")
            
            values.append(
                f"    ({carrera_id}, {area_id}, '{nom_carrera}', '{origina_objetivo}', '{objetivo_resumido}', '{pregunta_objetivo}')"
            )
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _extract_escuela_carrera(self) -> str:
        df = self.data['carrera_escuelas']
        sql_lines = [
            "-- ============================================",
            "-- INSERCIÓN DE RELACIÓN ESCUELA-CARRERA",
            "-- ============================================",
            "",
            "BEGIN;",
            "",
            "INSERT INTO public.escuela_carrera (id_escuela, id_carrera) VALUES"
        ]
        
        values = []
        for _, row in df.iterrows():
            id_escuela = row['id_escuela']
            id_carrera = row['id_carrera']
            values.append(f"    ({id_escuela}, {id_carrera})")
        
        sql_lines.append(",\n".join(values) + ";")
        sql_lines.extend([
            "",
            "COMMIT;",
        ])
        
        return "\n".join(sql_lines)
    
    def _save_sql(self, sql_content_list: list):
        output_file = self.ruta_sql_output / f"insert_data.sql"

        header = [
            "-- ============================================",
            "-- SCRIPT DE INSERCIÓN DE DATOS - ATHENA",
            "-- Generado automáticamente",
            f"-- Archivo: {self.ruta_excel.name}",
            "-- ============================================",
            ""
        ]

        full_sql = "\n".join(header) + "\n\n" + "\n\n".join(sql_content_list)

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(full_sql)

        print(f"✅ SQL guardado: {output_file}")

## ⚙️ Configuración y Ejecución

### Rutas configuradas:

- **`RUTA_EXCEL`**: Ubicación del archivo Excel con los datos (`tablas_completas.xlsx`)
- **`RUTA_SQL_OUTPUT`**: Directorio donde se guardará el script SQL generado

### Mapeo de columnas:

#### 📍 areas
- `id_area` → `area_id`
- `nombre_area` → `nom_area`

#### 🏫 escuelas
- `id_escuela` → `escuela_id`
- `nombre_escuela` → `nom_escuela`

#### 🎓 carreras
- `id_carrera` → `carrera_id`
- `nombre_carrera` → `nom_carrera`
- `id_area` → `area_id`
- `objetivo_resumido` → `objetivo`
- `pregunta` → `pregunta_objetivo`
- `objetivo` (largo) → `origina_objetivo`

#### 🔗 escuela_carrera (relación M:N)
- `id_escuela` → `id_escuela`
- `id_carrera` → `id_carrera`

### Ejecución:

Al ejecutar la celda siguiente, se:
1. Crea el directorio de output si no existe
2. Instancia el Extractor
3. Procesa todas las hojas del Excel
4. Genera el archivo `insert_data.sql` completo

In [12]:
RUTA_EXCEL = "../assets/layouts/excel/tablas_completas.xlsx"
RUTA_SQL_OUTPUT = "/Users/jmonroy/Documents/MyProjects/Athena/DataBase/scripts"
if __name__ == "__main__":
    Path(RUTA_SQL_OUTPUT).mkdir(parents=True, exist_ok=True)
    extractor = Extractor(RUTA_EXCEL, RUTA_SQL_OUTPUT)
    extractor.extract()

📊 Excel cargado: ../assets/layouts/excel/tablas_completas.xlsx
📁 Output SQL: /Users/jmonroy/Documents/MyProjects/Athena/DataBase/scripts
📋 Hojas disponibles: ['area', 'carreras', 'escuelas', 'carrera_escuelas']
✅ SQL guardado: /Users/jmonroy/Documents/MyProjects/Athena/DataBase/scripts/insert_data.sql


## ✅ Resultado

Si la ejecución fue exitosa, se habrá generado el archivo:
```
/Users/jmonroy/Documents/MyProjects/Athena/DataBase/scripts/insert_data.sql
```

### 📊 Contenido del archivo generado:

1. **Encabezado** con metadata del origen
2. **Sección de áreas** (3 registros)
3. **Sección de escuelas** (26 registros)
4. **Sección de carreras** (66 registros)
5. **Sección de relaciones escuela-carrera** (79 registros)

### 🚀 Próximos pasos:

1. **Revisar el SQL generado** para verificar que los datos sean correctos
2. **Ejecutar en PostgreSQL**:
   ```bash
   psql -U usuario -d athena -f insert_data.sql
   ```
3. **Verificar la inserción**:
   ```sql
   SELECT COUNT(*) FROM areas;      -- Debe ser 3
   SELECT COUNT(*) FROM escuelas;   -- Debe ser 26
   SELECT COUNT(*) FROM carreras;   -- Debe ser 66
   ```

### 🔧 Para agregar más extractores:

1. Agregar un nuevo `case` en el método `extract()`
2. Implementar el método `_extract_nombre_tabla()`
3. Seguir el mismo patrón de formato SQL